# New York Taxi Fare Prediction — PySpark Version

Versi PySpark dari notebook prediksi tarif taksi NYC. Notebook ini mencakup:
- Exploratory Data Analysis (EDA)
- Feature Engineering
- ML Pipeline dengan PySpark MLlib
- Linear Regression, Random Forest, dan GBT Regression

## 1. Setup SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler, StandardScaler, OneHotEncoder, StringIndexer,
    Bucketizer, PolynomialExpansion
)
from pyspark.ml.regression import (
    LinearRegression, RandomForestRegressor, GBTRegressor
)
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

import matplotlib.pyplot as plt
%matplotlib inline

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("NY Taxi Fare Prediction") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/05 21:40:45 WARN Utils: Your hostname, aryas-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.37 instead (on interface en0)
26/05/05 21:40:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 21:40:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


## 2. Load Data

In [2]:
# Load 1 juta baris pertama dari train.csv
# PySpark membaca seluruh file secara distributed;
# gunakan .limit() untuk membatasi jumlah baris seperti nrows di pandas
train = spark.read.csv("dataset/train.csv", header=True, inferSchema=True).limit(1_000_000)

print(f"Jumlah baris: {train.count()}")
train.printSchema()

Jumlah baris: 1000000
root
 |-- key: timestamp (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- passenger_count: integer (nullable = true)



## 3. Exploratory Data Analysis

In [3]:
# Tampilkan 10 baris pertama
train.show(10, truncate=False)

+-------------------+-----------+-------------------+----------------+---------------+-----------------+----------------+---------------+
|key                |fare_amount|pickup_datetime    |pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|passenger_count|
+-------------------+-----------+-------------------+----------------+---------------+-----------------+----------------+---------------+
|2009-06-15 17:26:21|4.5        |2009-06-16 00:26:21|-73.844311      |40.721319      |-73.84161        |40.712278       |1              |
|2010-01-05 16:52:16|16.9       |2010-01-05 23:52:16|-74.016048      |40.711303      |-73.979268       |40.782004       |1              |
|2011-08-18 00:35:00|5.7        |2011-08-18 07:35:00|-73.982738      |40.76127       |-73.991242       |40.750562       |2              |
|2012-04-21 04:30:42|7.7        |2012-04-21 11:30:42|-73.98713       |40.733143      |-73.991567       |40.758092       |1              |
|2010-03-09 07:51:00|5.3        |2

In [4]:
# Statistik deskriptif
train.describe().show()

26/05/05 21:41:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+------------------+------------------+------------------+-----------------+------------------+
|summary|      fare_amount|  pickup_longitude|   pickup_latitude| dropoff_longitude| dropoff_latitude|   passenger_count|
+-------+-----------------+------------------+------------------+------------------+-----------------+------------------+
|  count|          1000000|           1000000|           1000000|            999990|           999990|           1000000|
|   mean|11.34807872000041|-72.52663992960902|39.929007750361336|-72.52785952327304|39.91995441227741|          1.684924|
| stddev|9.822090134942536|12.057936827647252| 7.626154182522038|11.324494381303207|8.201417645775152|1.3239112005587288|
|    min|            -44.9|      -3377.680935|      -3116.285383|      -3383.296608|     -3114.338567|                 0|
|    max|            500.0|       2522.271325|        2621.62843|         45.581619|      1651.553433|               208|
+-------+---------------

In [6]:
from pyspark.sql.functions import col, when, count, isnan
from pyspark.sql.types import NumericType

train.select([
    count(
        when(
            col(c).isNull() | (isnan(col(c)) if isinstance(train.schema[c].dataType, NumericType) else False),
            c
        )
    ).alias(c)
    for c in train.columns
]).show()

+---+-----------+---------------+----------------+---------------+-----------------+----------------+---------------+
|key|fare_amount|pickup_datetime|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|passenger_count|
+---+-----------+---------------+----------------+---------------+-----------------+----------------+---------------+
|  0|          0|              0|               0|              0|               10|              10|              0|
+---+-----------+---------------+----------------+---------------+-----------------+----------------+---------------+



In [ ]:
# Ekstrak fitur waktu dari pickup_datetime
train = train.withColumn("pickup_datetime", F.to_timestamp("pickup_datetime")) \
             .withColumn("pickup_year",  F.year("pickup_datetime").cast(DoubleType())) \
             .withColumn("pickup_dow",   F.dayofweek("pickup_datetime").cast(IntegerType())) \
             .withColumn("pickup_hour",  F.hour("pickup_datetime").cast(IntegerType()))

train.select("pickup_datetime", "pickup_year", "pickup_dow", "pickup_hour").show(5)

In [ ]:
# Rata-rata fare per tahun — visualisasi via pandas (collect ke driver)
fare_by_year = train.groupBy("pickup_year").agg(
    F.mean("fare_amount").alias("avg_fare")
).orderBy("pickup_year").toPandas()

fare_by_year.plot(x="pickup_year", y="avg_fare", marker="o", title="Rata-rata Tarif per Tahun")
plt.xlabel("Tahun")
plt.ylabel("Rata-rata Tarif ($)")
plt.tight_layout()
plt.show()

In [ ]:
# Rata-rata fare per hari dalam seminggu
fare_by_dow = train.groupBy("pickup_dow").agg(
    F.mean("fare_amount").alias("avg_fare")
).orderBy("pickup_dow").toPandas()

fare_by_dow.plot(x="pickup_dow", y="avg_fare", marker="o", title="Rata-rata Tarif per Hari")
plt.xlabel("Hari (1=Minggu, 7=Sabtu)")
plt.ylabel("Rata-rata Tarif ($)")
plt.tight_layout()
plt.show()

In [ ]:
# Rata-rata fare per jam
fare_by_hour = train.groupBy("pickup_hour").agg(
    F.mean("fare_amount").alias("avg_fare")
).orderBy("pickup_hour").toPandas()

fare_by_hour.plot(x="pickup_hour", y="avg_fare", marker="o", title="Rata-rata Tarif per Jam")
plt.xlabel("Jam")
plt.ylabel("Rata-rata Tarif ($)")
plt.tight_layout()
plt.show()

## 4. Feature Engineering — Jarak Tempuh

In [ ]:
# Hitung jarak garis lurus (approximasi dalam km)
# 1 derajat latitude ≈ 111 km, 1 derajat longitude ≈ 85 km di NYC
train = train \
    .withColumn("move_latitude",
        (F.col("dropoff_latitude") - F.col("pickup_latitude")) * 111) \
    .withColumn("move_longitude",
        (F.col("dropoff_longitude") - F.col("pickup_longitude")) * 85) \
    .withColumn("abs_distance",
        F.sqrt(F.col("move_latitude")**2 + F.col("move_longitude")**2))

# Arah perjalanan (fitur boolean → integer untuk PySpark)
train = train \
    .withColumn("toward_east",  (F.col("move_longitude") > 0).cast(IntegerType())) \
    .withColumn("toward_north", (F.col("move_latitude")  > 0).cast(IntegerType()))

train.select("abs_distance", "toward_east", "toward_north").show(5)

## 5. Data Cleaning — Filter Data Abnormal

In [ ]:
# Filter:
#   a. fare_amount > 0
#   b. koordinat pickup dalam radius 1 derajat dari NYC (-73.9, 40.7)
#   c. abs_distance antara 0 dan 100 km

train_trim = train.filter(
    (F.col("fare_amount") > 0) &
    (F.col("pickup_longitude").between(-74.4, -73.4)) &
    (F.col("pickup_latitude").between(40.2, 41.2)) &
    (F.col("abs_distance") > 0) &
    (F.col("abs_distance") < 100)
)

total     = train.count()
trimmed   = train_trim.count()
print(f"Data awal   : {total:,}")
print(f"Setelah trim: {trimmed:,}")
print(f"Rasio       : {trimmed/total:.4f} ({trimmed/total*100:.2f}%)")

In [ ]:
# Korelasi fitur numerik terhadap fare_amount
num_cols = ["fare_amount", "pickup_year", "abs_distance",
            "pickup_longitude", "dropoff_longitude",
            "pickup_latitude",  "dropoff_latitude"]

from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
import pandas as pd
import numpy as np

assembler_corr = VectorAssembler(inputCols=num_cols, outputCol="corr_features", handleInvalid="skip")
corr_df = assembler_corr.transform(train_trim)

corr_matrix = Correlation.corr(corr_df, "corr_features").head()[0].toArray()
corr_series = pd.Series(corr_matrix[0], index=num_cols, name="correlation_with_fare_amount")
print(corr_series.sort_values(ascending=False))

## 6. Preprocessing Pipeline (PySpark MLlib)

Fitur yang digunakan:
- **Kategorik**: `pickup_hour` (di-bucket), `pickup_dow` → OneHotEncoding
- **Numerik**: `pickup_year`, `abs_distance`, `pickup_longitude`, `dropoff_longitude` → StandardScaler

In [ ]:
# ── Bucketizer untuk pickup_hour (8 bin, seperti pd.cut di versi pandas) ──
# Batas bin: 0,3,6,9,12,15,18,21,24
hour_splits = [0.0, 3.0, 6.0, 9.0, 12.0, 15.0, 18.0, 21.0, 24.0]
hour_bucketizer = Bucketizer(
    splits=hour_splits,
    inputCol="pickup_hour",
    outputCol="pickup_hour_bucket",
    handleInvalid="keep"
)

# ── OneHotEncoder untuk hour bucket dan pickup_dow ──
ohe = OneHotEncoder(
    inputCols=["pickup_hour_bucket", "pickup_dow"],
    outputCols=["pickup_hour_ohe",   "pickup_dow_ohe"],
    handleInvalid="keep"
)

# ── Assembler & Scaler untuk fitur numerik ──
num_features = ["pickup_year", "abs_distance", "pickup_longitude", "dropoff_longitude"]

num_assembler = VectorAssembler(
    inputCols=num_features,
    outputCol="num_raw",
    handleInvalid="skip"
)
scaler = StandardScaler(
    inputCol="num_raw",
    outputCol="num_scaled",
    withStd=True, withMean=True
)

# ── Gabungkan semua fitur menjadi satu vector ──
final_assembler = VectorAssembler(
    inputCols=["pickup_hour_ohe", "pickup_dow_ohe", "num_scaled"],
    outputCol="features",
    handleInvalid="skip"
)

# Preprocessing pipeline bersama (tanpa model)
preprocessing_stages = [hour_bucketizer, ohe, num_assembler, scaler, final_assembler]

print("Preprocessing pipeline siap.")

In [ ]:
# Siapkan kolom label (fare_amount harus bertipe Double)
train_trim = train_trim.withColumn("label", F.col("fare_amount").cast(DoubleType()))

# Train/test split — 80/20
train_df, test_df = train_trim.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count():,} | Test: {test_df.count():,}")

## 7. Linear Regression

In [ ]:
lr = LinearRegression(featuresCol="features", labelCol="label", maxIter=100)

lr_pipeline = Pipeline(stages=preprocessing_stages + [lr])
lr_model    = lr_pipeline.fit(train_df)

evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")

lr_pred = lr_model.transform(test_df)
lr_rmse = evaluator.evaluate(lr_pred)
print(f"Linear Regression RMSE: {lr_rmse:.4f}")

## 8. Polynomial Regression (via PolynomialExpansion)

In [ ]:
# PySpark menyediakan PolynomialExpansion sebagai pengganti PolynomialFeatures sklearn
# Terapkan hanya pada fitur numerik sebelum assembler akhir

poly_num_assembler = VectorAssembler(
    inputCols=num_features,
    outputCol="num_raw_poly",
    handleInvalid="skip"
)
poly_scaler_pre = StandardScaler(
    inputCol="num_raw_poly", outputCol="num_pre_poly",
    withStd=True, withMean=True
)
poly_exp = PolynomialExpansion(
    degree=2,              # coba degree 2 atau 3
    inputCol="num_pre_poly",
    outputCol="num_poly"
)
poly_scaler_post = StandardScaler(
    inputCol="num_poly", outputCol="num_scaled_poly",
    withStd=True, withMean=True
)
poly_final_assembler = VectorAssembler(
    inputCols=["pickup_hour_ohe", "pickup_dow_ohe", "num_scaled_poly"],
    outputCol="features",
    handleInvalid="skip"
)

poly_stages = [
    hour_bucketizer, ohe,
    poly_num_assembler, poly_scaler_pre, poly_exp, poly_scaler_post,
    poly_final_assembler
]

poly_lr = LinearRegression(featuresCol="features", labelCol="label", maxIter=100)
poly_pipeline = Pipeline(stages=poly_stages + [poly_lr])
poly_model    = poly_pipeline.fit(train_df)

poly_pred = poly_model.transform(test_df)
poly_rmse = evaluator.evaluate(poly_pred)
print(f"Polynomial Regression (degree=2) RMSE: {poly_rmse:.4f}")

## 9. Random Forest Regression

In [ ]:
rf = RandomForestRegressor(
    featuresCol="features", labelCol="label",
    numTrees=20, maxDepth=8, seed=42
)

rf_pipeline = Pipeline(stages=preprocessing_stages + [rf])
rf_model    = rf_pipeline.fit(train_df)

rf_pred = rf_model.transform(test_df)
rf_rmse = evaluator.evaluate(rf_pred)
print(f"Random Forest RMSE: {rf_rmse:.4f}")

In [ ]:
# ── Hyperparameter Tuning — Random Forest via CrossValidator ──
param_grid_rf = ParamGridBuilder() \
    .addGrid(rf.numTrees,  [10, 20, 50]) \
    .addGrid(rf.maxDepth,  [6, 8, 10]) \
    .build()

cv_rf = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=param_grid_rf,
    evaluator=evaluator,
    numFolds=5,
    seed=42
)

cv_rf_model = cv_rf.fit(train_df)
best_rf_pred = cv_rf_model.transform(test_df)
best_rf_rmse = evaluator.evaluate(best_rf_pred)
print(f"Best Random Forest RMSE (after CV): {best_rf_rmse:.4f}")

# Parameter terbaik
best_rf = cv_rf_model.bestModel.stages[-1]
print(f"numTrees terbaik : {best_rf.getNumTrees}")
print(f"maxDepth terbaik : {best_rf.getOrDefault('maxDepth')}")

## 10. Gradient Boosted Trees (GBT) — Pengganti Non-linear SVR

In [ ]:
# GBT lebih cocok di PySpark dibanding SVR untuk dataset besar
gbt = GBTRegressor(
    featuresCol="features", labelCol="label",
    maxIter=50, maxDepth=6, seed=42
)

gbt_pipeline = Pipeline(stages=preprocessing_stages + [gbt])
gbt_model    = gbt_pipeline.fit(train_df)

gbt_pred = gbt_model.transform(test_df)
gbt_rmse = evaluator.evaluate(gbt_pred)
print(f"GBT Regression RMSE: {gbt_rmse:.4f}")

## 11. Perbandingan Model

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["Linear Regression", "Polynomial Regression (deg=2)",
              "Random Forest (default)", "Random Forest (tuned)",
              "GBT Regression"],
    "RMSE":  [lr_rmse, poly_rmse, rf_rmse, best_rf_rmse, gbt_rmse]
}).sort_values("RMSE")

print(results.to_string(index=False))

results.plot(x="Model", y="RMSE", kind="barh", figsize=(9, 4),
             title="Perbandingan RMSE Antar Model")
plt.xlabel("RMSE ($)")
plt.tight_layout()
plt.show()

## 12. Prediksi pada Data Test

In [ ]:
# Load test.csv
test = spark.read.csv("dataset/test.csv", header=True, inferSchema=True)

# Feature engineering yang sama dengan training
test = test \
    .withColumn("pickup_datetime", F.to_timestamp("pickup_datetime")) \
    .withColumn("pickup_year",  F.year("pickup_datetime").cast(DoubleType())) \
    .withColumn("pickup_dow",   F.dayofweek("pickup_datetime").cast(IntegerType())) \
    .withColumn("pickup_hour",  F.hour("pickup_datetime").cast(IntegerType())) \
    .withColumn("move_latitude",
        (F.col("dropoff_latitude") - F.col("pickup_latitude")) * 111) \
    .withColumn("move_longitude",
        (F.col("dropoff_longitude") - F.col("pickup_longitude")) * 85) \
    .withColumn("abs_distance",
        F.sqrt(F.col("move_latitude")**2 + F.col("move_longitude")**2))

test.printSchema()
test.show(5)

In [ ]:
# Gunakan model terbaik (Random Forest tuned) untuk prediksi
final_model  = cv_rf_model   # ganti dengan gbt_model jika GBT lebih baik
test_pred    = final_model.transform(test)

# Tampilkan hasil prediksi
test_pred.select("key", "pickup_datetime", "prediction").show(10)

In [ ]:
# Simpan hasil prediksi ke CSV
test_pred.select(
    F.col("key"),
    F.col("prediction").alias("fare_amount")
).write.csv("submission_pyspark", header=True, mode="overwrite")

print("Prediksi tersimpan di folder 'submission_pyspark/'")

In [ ]:
# Stop SparkSession
spark.stop()
print("SparkSession dihentikan.")